# NUTDTS 816 Time Series Analysis
## L19 VAR and Granger causality

Lab notebook for Chapter 10 of the lecture notes. Run the setup cell first. Every code cell reproduces an example from the notes; the exercises at the end are from the chapter's self-check list.

**Instructor:** Dr Tolulope Adesina · NUTM MSc Data Science · 2026

In [ ]:
# ---- Setup: run once per Colab session ----
# 1. Install the course libraries (about two minutes the first time)
!pip install -q statsmodels pmdarima statsforecast neuralforecast lightgbm arch plotly

# 2. Fetch the course data module and data snapshots from the course repository.
#    Replace REPO with your fork or the official course repository URL.
REPO = "https://raw.githubusercontent.com/<your-github-user>/nutdts816/main"
import urllib.request, os
os.makedirs("data", exist_ok=True)
urllib.request.urlretrieve(f"{REPO}/src/tsdata.py", "tsdata.py")
DATA_FILES = ["nigeria_cpi", "nigeria_fx", "nigeria_grid", "nigeria_malaria", "nigeria_rainfall", "bonny_light", "daily_demand",
              "airpassengers", "a10", "h02", "ausbeer", "elecequip", "usmelec", "goog", "nile", "austourists", "oil", "dax", "uschange", "elecdemand"]
for f in DATA_FILES:
    urllib.request.urlretrieve(f"{REPO}/data/{f}.csv", f"data/{f}.csv")

import warnings; warnings.filterwarnings("ignore")
import pandas as pd, numpy as np, matplotlib.pyplot as plt
plt.rcParams.update({"figure.figsize": (9, 3.6), "axes.grid": True, "grid.alpha": 0.3, "axes.spines.top": False, "axes.spines.right": False})
import tsdata
print("Setup complete.")

### 10.2 Worked example: a VAR for US consumption, income and production growth

In [ ]:
import pandas as pd, numpy as np, matplotlib.pyplot as plt
from statsmodels.tsa.api import VAR
import tsdata
us = tsdata.uschange()[['Consumption', 'Income', 'Production']]
us = us.asfreq('QS')
train, test = us[:'2014-12'], us['2015-01':]          # hold out the last 8 quarters
model = VAR(train)
print(model.select_order(maxlags=8).summary())

In [ ]:
var3 = model.fit(3)
print('Stable (all companion eigenvalues inside unit circle):', var3.is_stable(), '| smallest inverse-root modulus (must exceed 1):', np.round(np.abs(var3.roots).min(), 3))
print('\nEquation for Consumption (growth in consumption depends on lags of all three):')
print(var3.params['Consumption'].round(3).to_string())
w = var3.test_whiteness(nlags=12); print(f'\nResidual Portmanteau test up to lag 12 (H0: no residual autocorrelation): statistic = {w.test_statistic:.1f}, p = {w.pvalue:.3f}')

In [ ]:
h = len(test)
fc = var3.forecast(train.values[-3:], steps=h); fc = pd.DataFrame(fc, index=test.index, columns=us.columns)
lo, mid, hi = var3.forecast_interval(train.values[-3:], steps=h, alpha=0.2)
fig, axes = plt.subplots(1, 3, figsize=(11, 3))
for i, col in enumerate(us.columns):
    us[col]['2008':].plot(ax=axes[i], lw=1, label='observed'); fc[col].plot(ax=axes[i], lw=2, color='#B8860B', label='VAR(3) forecast')
    axes[i].fill_between(test.index, lo[:, i], hi[:, i], color='#B8860B', alpha=0.2); axes[i].set_title(col + ' growth (%)'); axes[i].set_xlabel('')
axes[0].legend(fontsize=8)
mae_var = (fc - test).abs().mean(); mae_mean = (train.iloc[-20:].mean() - test).abs().mean()
print(pd.DataFrame({'VAR(3) MAE': mae_var, 'Recent-mean MAE': mae_mean}).round(3).to_string())
_caption = 'Eight-quarter VAR forecasts revert quickly to the long-run means. Growth rates are only modestly forecastable: on this hold-out the VAR ties a recent mean for income and loses narrowly for consumption and production. Quarterly growth rates are close to unforecastable beyond a quarter or two, and a VAR cannot change that.'

### 10.3 Granger causality

In [ ]:
rows = []
for caused in us.columns:
    for causing in us.columns:
        if caused != causing:
            r = var3.test_causality(caused, [causing], kind='f'); rows.append({'does': causing, 'Granger-cause': caused, 'F': round(r.test_statistic, 2), 'p-value': round(r.pvalue, 3)})
print(pd.DataFrame(rows).to_string(index=False))

### 10.4 Impulse responses and variance decomposition

In [ ]:
irf = var3.irf(10)
fig = irf.plot(orth=True, impulse='Consumption', figsize=(9, 6))
fig.suptitle('Orthogonalised impulse responses to a one-s.d. shock in Consumption growth (ordering: Consumption, Income, Production)', y=1.0)
_caption = 'A positive consumption shock raises income and production growth over the following two or three quarters; the effects die out within a year, as expected for growth rates. The shaded bands are 95% bootstrap intervals.'

In [ ]:
fevd = var3.fevd(8)
print('Share of 8-quarter forecast error variance of each series due to each shock:')
print(pd.DataFrame(fevd.decomp[:, -1, :], index=us.columns, columns=[f'{c} shock' for c in us.columns]).round(3).to_string())

## Exercises

1. Write out the VAR(1) for three series in full and count its parameters. How many observations would you want before estimating a VAR(4) with three series?
2. Re-estimate the US VAR with $p = 1$ and $p = 4$; report the Granger-causality results for income to consumption in each. Is the conclusion robust?
3. Using the simulated `nigeria_malaria()` weekly cases and `nigeria_rainfall()` (resampled to weekly by interpolation), test whether rainfall Granger-causes cases, and at what lag. What would you need to check before trusting the result?

In [ ]:
# Your work here
